# Why Agents?

What is an agent?

An agent is a reference to combining models that can perform some kind of reasoning, like large language models (e.g ChatGPT, Llama2, Mistral, etc...) with tools to give it access to the real world,
so they can do things like browsing the internet, buying stuff, etc...

Ok, so, why is there so much hype around agents right now?

Because Agents are cool! Recently with the advance of LLMs, we've seen them become an amazing tool to do all sorts of things like building apps, browse the internet and more.



SOme neat examples of these kinds of agents can be found in here:

- [AutoGPT](https://github.com/Significant-Gravitas/AutoGPT)
- [GPT-Engineer](https://github.com/gpt-engineer-org/gpt-engineer)
- [BabyAGI](https://github.com/yoheinakajima/babyagi)

Now, today although they seem extremely powerful, agents are still at a very early stage in terms of readiness to deploy as products, something you can atest by listening to Andrej Karpathy talk about agents in this talk here:

- [Karpathy on Agents](https://www.youtube.com/watch?v=fqVLjtvWgq8)

This live-training is all about! Getting you excited about this amazing new technology, understanding it from the ground up but with a focus on practical applications and fun stuff you can do with them! 

# What is an Agent?

An agent is nothing more than some entity that can _think_ and _act_, that's right, in a way you're an agent! 

After all you can think and act on those thoughts like in the case of coming to this live-training:

- Thought: "I want to learn about agents"

- Action: "Go to the internet and research cool platforms where I can learn about agents"

- Thought: "O'Reilly has some awesome courses and live-trainings"

- Action: "Look up O'Reilly courses"

- Thought: "Live-trainings by instructor Lucas are awesome"

- Action: "Schedule live-training about agents with instructor Lucas Soares" (lol)

In a way this is a simplified rendition of what brought you here, obviously not necessarily in this particular order nor these particular sets of thought and action pairs. This particular way of thinking about how to structure thoughts and actions is well represented in the paper: [ReACT](https://arxiv.org/pdf/2210.03629.pdf). 

With regards to LLMs, how can bring this idea to fruition thinking about the LLM model as the reasoning and thinking engine?

We can start simple and just call the openai API to start:

In [1]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
# uncomment this if running locally
#!pip install python-dotenv
from dotenv import load_dotenv

load_dotenv()

import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("OPENAI_API_KEY")

In [3]:
from openai import OpenAI
from IPython.display import Markdown

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="You are a helpful research and programming assistant.",
    input="Create a joke about a bald teacher explaining agents to wonderful students.",
)

Markdown(response.output_text)


Why did the bald teacher love explaining agents to his wonderful students?

Because even without hair, he knew how to *act* and always *agent-ly* kept their attention!

Ok cool, so here we have three ideas of actions to perform:

- Creating directories
- Listing files
- Removing files

Let's transform them into functions that we could call just like in any type of Python-based application.

In [4]:
def create_directory(dir_name):
    os.makedirs(dir_name, exist_ok=True)

def create_file(filename):
    with open(filename, 'w'):
        pass

def list_files():
    files = os.listdir()
    for file in files:
        print(file)

Now, let's imagine that we wanted to create an agent that would perform these actions for us based on some input that we give it, how can we connect models that we know and can use today like ChatGPT, with these tools that do stuff in the real world?

Before the major LLM providers shipped structured tool-calling APIs, the trick was to *prompt* the model to output a function call as text, then parse and execute that text in your own code. This is the **"old way"** — fragile, but worth seeing because it makes the core idea concrete: the LLM acts as a planner that picks which function to call.

The famous paper ['Toolformer'](https://arxiv.org/pdf/2302.04761.pdf) showed that LLMs can teach themselves to call external tools.

Below we hack our way into wiring an LLM up to a few Python functions — keeping in mind that **today you'd reach for the structured tool-calling API instead**, which we'll cover in the next section.


Now, how can we actually put it all together so that given a task, a model can:

- Plan the task
- Execute actions to complete the task
- Know when to call a function


This is actually an interesting problem, let's understand why is that the case by trying to hack our way into putting all of these together:

In [5]:
# Old-way demo: ask the model to *emit* a function call as text, then exec() it.
# This works, but it's fragile — the entire system depends on the model's
# formatting discipline. We'll fix this properly with structured tool calling
# in the next section.

task_description = (
    "Create a folder called 'funny-pancakes-recipes'. "
    "Inside that folder, create a file called '3-funny-pancake-recipes.md'."
)

prompt = f"""Given this task: {task_description}

You have access to the following functions:

    def create_directory(dir_name): ...
    def create_file(filename): ...
    def list_files(): ...

Output ONLY the first Python function call needed to start the task.
No prose, no markdown fences, just the call.
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="You are a helpful research and programming assistant.",
    input=prompt,
)

output = response.output_text
Markdown(output)


create_directory('funny-pancakes-recipes')

In [6]:
import os
# Remove any ```python or ``` tags from the output string if present
output_clean = output.replace("```python", "").replace("```", "").strip()
Markdown(output_clean)

create_directory('funny-pancakes-recipes')

In [7]:
exec(output_clean)

In [8]:
!ls -d ./* | grep pancakes

./funny-pancakes-recipes
./pancakes-are-better-than-waffles
./pancakes-philosophy.txt


At this point we can start identifying a lot of issues with this approach despite our early sucess:

- Uncertainty of model's outputs can affect our ability to reliably call the functions
- We need more structured ways to prepare the inputs of the function calls
- We need better ways to put everything together (just feeding the entire functions like this makes it a very clunky and non-scalable framework for more complex cases)

There are many more issues but starting with these, we can now look at frameworks and see how they fix these issues and with that in mind understand what is behind their implementations!

I personally think this is a much better way to understand what is going on behind agents in practice rather than just use the more higher level frameworks right of the bat!

# OpenAI Functions

Ok, let's see how [OpenAI](https://openai.com/) lets us connect models to tools properly — without prompt-hacking.

The modern way is the [**Responses API**](https://platform.openai.com/docs/api-reference/responses) with structured tool calling. The flow is:

1. Call the model with the user input and a list of tools.
2. The model may emit one or more `function_call` items in its output, each with a `name`, `arguments` (a JSON string), and `call_id`.
3. Your code parses the arguments, runs the real function, and appends a `function_call_output` item (matched to `call_id`) to the conversation.
4. You call the model again so it can summarize the result back to the user.

No regex, no `exec()`, no parsing tricks.


Below is an example taken from their official documentation:

Let's look at how our previous model with those three simple functions: `create_directory()`, `create_file()`, and `list_files()` would be implemented using OpenAI's function calling approach:

In [9]:
import json
import os


def create_directory(directory_name):
    """Function that creates a directory given a directory name."""
    if not os.path.exists(directory_name):
        os.mkdir(directory_name)
        return json.dumps({"directory_name": directory_name})
    else:
        return json.dumps({"directory_name": directory_name, "error": "Directory already exists"})


# Responses API tool format: flat fields, no nested "function" wrapper.
tool_create_directory = {
    "type": "function",
    "name": "create_directory",
    "description": "Create a directory given a directory name.",
    "parameters": {
        "type": "object",
        "properties": {
            "directory_name": {
                "type": "string",
                "description": "The name of the directory to create.",
            }
        },
        "required": ["directory_name"],
    },
}

tools = [tool_create_directory]


In [10]:
import json


def run_terminal_task():
    user_msg = "Create a folder called 'pancakes-are-better-than-waffles'."
    input_list = [{"role": "user", "content": user_msg}]

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=input_list,
        tools=tools,
    )

    available_functions = {"create_directory": create_directory}

    # In the Responses API, function calls are first-class items in `response.output`.
    function_calls = [item for item in response.output if item.type == "function_call"]

    if not function_calls:
        return response

    # Append the model's output items, then one `function_call_output` per call.
    input_list += response.output
    for call in function_calls:
        fn = available_functions[call.name]
        args = json.loads(call.arguments)
        result = fn(**args)
        input_list.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": result,
        })

    # Second call: model now sees the tool outputs and can summarize for the user.
    final = client.responses.create(
        model="gpt-4.1-mini",
        input=input_list,
        tools=tools,
    )
    return final


output = run_terminal_task()
output


Response(id='resp_0e1f48f9a0c507a6006a0592fd53a88197a3dd515e467dd382', created_at=1778750205.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_0e1f48f9a0c507a6006a0592fe0a708197895aa5391509db67', content=[ResponseOutputText(annotations=[], text="The folder 'pancakes-are-better-than-waffles' already exists. Is there anything else you would like to do with this folder?", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='create_directory', parameters={'type': 'object', 'properties': {'directory_name': {'type': 'string', 'description': 'The name of the directory to create.'}}, 'required': ['directory_name'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Create a directory given a directory na

In [11]:
print(output.output_text)


The folder 'pancakes-are-better-than-waffles' already exists. Is there anything else you would like to do with this folder?


In [12]:
!ls -d */ | grep waffles

pancakes-are-better-than-waffles/


Great! We implemented openai function calling for creating directories! We could evolve this approach but let's stop for now.

See more info on these examples from OpenAI's [official cookbook](https://cookbook.openai.com/examples/how_to_call_functions_with_chat_models).

Now, let's implement the agent loop logic ourselves — in pure Python on top of the OpenAI Responses API. No LangChain, no LangGraph. We'll cover those higher-level frameworks later; the goal here is to see every moving part of an agent.

The pieces we need:

- **The LLM** — a thin wrapper around `client.responses.create` that holds its own conversation history and tool definitions.
- **Tools** — plain Python functions plus a small JSON schema each, so the model knows how to call them.
- **The loop** — keep stepping the model until it stops asking for tool calls (or we hit a safety cap on iterations).


**The LLM**

In [13]:
import json
from openai import OpenAI


class LLM:
    """A small wrapper around `client.responses.create` that:
      - keeps its own conversation `history` (a list of Responses-API input items),
      - executes any function_call items the model emits,
      - appends `function_call_output` items back to history so the next turn sees them.
    """

    def __init__(self, model="gpt-4.1-mini", tools=None, tool_functions=None, system_prompt=""):
        self.model = model
        self.client = OpenAI()
        self.system_prompt = system_prompt
        self.tools = tools or []
        self.tool_functions = tool_functions or {}
        self.history = []  # list of Responses-API input items

    def _call_model(self):
        return self.client.responses.create(
            model=self.model,
            instructions=self.system_prompt or None,
            input=self.history,
            tools=self.tools or None,
        )

    def call(self, user_message=None):
        """Run one model turn.

        If `user_message` is provided, it's appended as a user input item first.
        Any function_call items the model emits are executed here, and their outputs
        are appended to `history`. Returns `(response, function_calls)` so the caller
        can decide whether to take another turn.
        """
        if user_message is not None:
            self.history.append({"role": "user", "content": user_message})

        response = self._call_model()

        # Persist *all* output items (messages, function_calls, ...) so the model
        # sees its own prior calls on the next turn.
        self.history += response.output

        function_calls = [item for item in response.output if item.type == "function_call"]
        for call in function_calls:
            fn = self.tool_functions.get(call.name)
            if fn is None:
                result = f"Error: tool {call.name!r} is not registered."
            else:
                try:
                    args = json.loads(call.arguments) if call.arguments else {}
                    result = fn(**args)
                except Exception as e:
                    result = f"Error executing {call.name!r}: {e}"

            self.history.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": str(result),
            })

        return response, function_calls


# Sanity check: a no-tools LLM should just return text.
demo_llm = LLM(model="gpt-4.1-mini", system_prompt="You are a helpful assistant.")
response, _ = demo_llm.call("Create a one-line joke about a bald teacher explaining agents.")
print(response.output_text)


Why did the bald teacher make a great agent? Because he always knew how to handle agents without any "hair-itage" issues!


**Tools**

In [14]:
import os

def create_dir(folder_path):
    """
    Creates a directory given a folder path.
    """
    if not os.path.exists(folder_path):
        os.mkdir(folder_path)
    
    return f"Folder path was created at: {folder_path}"


create_dir("lucas-test-local-agent")

'Folder path was created at: lucas-test-local-agent'

In [15]:
!ls -d */ | grep lucas-test-local-agent

lucas-test-local-agent/


In [16]:
def create_file(file_path, contents=""):
    """
    Creates a file with content.
    If no content is provided it will create an empty file.
    """
    
    with open(file_path, "w") as f:
        f.write(contents)
    
    return f"A file was created at: {file_path}"

create_file("./lucas-test-local-agent/file-text.txt", "This is a test")

'A file was created at: ./lucas-test-local-agent/file-text.txt'

In [17]:
def read_file(file_path):
    """
    Reads from file given its path.
    """
    
    with open(file_path, "r") as f:
        contents = f.read()
    
    return contents

read_file("./lucas-test-local-agent/file-text.txt")

'This is a test'

In [18]:
# Tool schemas for the Responses API. Flat shape: name/description/parameters at top level.
tool_schemas = [
    {
        "type": "function",
        "name": "create_dir",
        "description": "Create a directory at the given path.",
        "parameters": {
            "type": "object",
            "properties": {
                "folder_path": {"type": "string", "description": "Path of the folder to create."}
            },
            "required": ["folder_path"],
        },
    },
    {
        "type": "function",
        "name": "create_file",
        "description": "Create a file, optionally with text contents.",
        "parameters": {
            "type": "object",
            "properties": {
                "file_path": {"type": "string", "description": "Path of the file to create."},
                "contents": {"type": "string", "description": "Optional text contents."},
            },
            "required": ["file_path"],
        },
    },
    {
        "type": "function",
        "name": "read_file",
        "description": "Read and return the contents of a file at the given path.",
        "parameters": {
            "type": "object",
            "properties": {
                "file_path": {"type": "string", "description": "Path of the file to read."}
            },
            "required": ["file_path"],
        },
    },
]

# Name -> callable mapping the LLM class uses to actually execute calls.
tool_functions = {
    "create_dir": create_dir,
    "create_file": create_file,
    "read_file": read_file,
}


**Putting it all together**

We now have:

- `LLM`: a thin wrapper around `client.responses.create` that maintains its own conversation `history` and knows how to execute tool calls.
- `tool_schemas`: the JSON shapes the model sees.
- `tool_functions`: the real Python callables, looked up by name.

The agent loop just calls `llm.call()` repeatedly until the model stops asking for tools.


In [19]:
# Quick smoke test of the LLM class: one step, expect one tool call.
test_llm = LLM(
    model="gpt-4.1-mini",
    tools=tool_schemas,
    tool_functions=tool_functions,
    system_prompt="You are a desktop assistant. Use tools when the user asks for file/folder actions.",
)
response, function_calls = test_llm.call("Create a file named ./test-agent.txt")
print("Function calls issued this turn:")
for call in function_calls:
    print(f"  {call.name}({call.arguments})  call_id={call.call_id}")

Function calls issued this turn:
  create_file({"file_path":"./test-agent.txt","contents":""})  call_id=call_E7Bq6kIXeeAuHYAZOko7653j


In [20]:
def agent_loop(query, max_iters=5):
    """
    A minimal ReAct-style loop on top of the Responses API.

    Stopping conditions, in order:
      1. The model issued no function_call items this turn (it is done acting).
      2. We hit `max_iters` — a safety rail against runaway loops.
    """
    llm = LLM(
        model="gpt-4.1-mini",
        tools=tool_schemas,
        tool_functions=tool_functions,
        system_prompt=(
            "You are a desktop file-system assistant. "
            "Use the available tools to satisfy the user's request. "
            "When the task is complete, reply with a short summary and stop calling tools."
        ),
    )

    user_message = query
    response = None
    for i in range(max_iters):
        response, function_calls = llm.call(user_message=user_message)
        user_message = None  # only seed the user query on the first turn
        print(f"--- iter {i + 1}: {len(function_calls)} tool call(s) ---")
        for call in function_calls:
            print(f"  -> {call.name}({call.arguments})")

        if not function_calls:
            print("No new tool calls — agent finished.")
            return response.output_text
    print(f"Reached max iterations ({max_iters}) — stopping.")

    return response.output_text if response else ""


agent_loop(
    "Create a folder in current directory named 'testing-multiple-calls' "
    "and inside that folder create a file named nested-file.txt"
)

--- iter 1: 2 tool call(s) ---
  -> create_dir({"folder_path":"testing-multiple-calls"})
  -> create_file({"file_path":"testing-multiple-calls/nested-file.txt","contents":""})
--- iter 2: 0 tool call(s) ---
No new tool calls — agent finished.


"I have created a folder named 'testing-multiple-calls' in the current directory and inside that folder, I created a file named 'nested-file.txt'."

# References

- [HuggingGPT](https://github.com/microsoft/JARVIS)
- [Gen Agents](https://arxiv.org/pdf/2304.03442.pdf)
- [WebGPT](https://www.semanticscholar.org/paper/WebGPT%3A-Browser-assisted-question-answering-with-Nakano-Hilton/2f3efe44083af91cef562c1a3451eee2f8601d22)
- [LangChain](https://python.langchain.com/docs/get_started/introduction)
- [OpenAI](https://openai.com/)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
- [AutoGPT](https://github.com/Significant-Gravitas/AutoGPT)
- [GPT-Engineer](https://github.com/gpt-engineer-org/gpt-engineer)
- [BabyAGI](https://github.com/yoheinakajima/babyagi)
- [Karpathy on Agents](https://www.youtube.com/watch?v=fqVLjtvWgq8)
- [ReACT Paper](https://arxiv.org/abs/2210.03629)